In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis, pointbiserialr
from itertools import combinations

pd.set_option('display.max_columns', None)
sns.set(style="whitegrid", palette="pastel")

df = pd.read_csv("/home/onyxia/work/ml-uncertainty-quantification-and-sources/bank-full.csv", sep=',')

df.columns()

TypeError: 'Index' object is not callable

Échantillonnage stratifié : C’est un concept important souvent manqué lors du développement d’un modèle, que ce soit pour la régression ou la classification. Rappelez-vous que, pour éviter le surapprentissage de nos données, nous devons mettre en place une validation croisée, cependant, nous devons nous assurer qu’au moins les fonctionnalités ayant le plus d’influence sur notre étiquette (qu’un client potentiel ouvre un dépôt à terme ou non) soient réparties équitablement. Que veux-je dire par là ?

1) Nous devons voir comment nos données sont distribuées.
2) Après avoir noté que la colonne « prêt » contient 87 % de « non » (Ne pas de prêts personnels) et 13 % de « oui » (Avoir des prêts personnels).
3) Nous voulons nous assurer que notre ensemble d’entraînement et de test contient le même ratio de 87 % de « non » et 13 % de « oui ». Échantillonnage stratifié : C’est un concept important souvent manqué lors du développement d’un modèle, que ce soit pour la régression ou la classification.

Afin de préserver la distribution déséquilibrée de la variable cible, les données ont été séparées en ensembles d’apprentissage et de test à l’aide d’un échantillonnage stratifié basé sur la variable de souscription. Cette approche garantit une représentativité cohérente des classes dans les deux ensembles.

In [11]:
# Echantillonage train-set 

from sklearn.model_selection import StratifiedShuffleSplit

# Définition du split stratifié sur la variable cible
stratified = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

# Split
for train_idx, test_idx in stratified.split(df, df["y"]):
    stratified_train = df.loc[train_idx]
    stratified_test = df.loc[test_idx]

# Vérification des distributions
print("Distribution globale :")
print(df["y"].value_counts(normalize=True))

print("\nDistribution train :")
print(stratified_train["y"].value_counts(normalize=True))

print("\nDistribution test :")
print(stratified_test["y"].value_counts(normalize=True))


Distribution globale :
y
no     0.883015
yes    0.116985
Name: proportion, dtype: float64

Distribution train :
y
no     0.883018
yes    0.116982
Name: proportion, dtype: float64

Distribution test :
y
no     0.883003
yes    0.116997
Name: proportion, dtype: float64


In [12]:
# Encodage des variables
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer


categorical_nominal = [
    "job", "marital", "education",
    "contact", "month", "poutcome"
]

categorical_binary = ["default", "housing", "loan"]

numerical_features = [
    "age", "balance", "day",
    "campaign", "pdays", "previous",
]

#making preprocesseur
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat_nom", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_nominal),
        ("cat_bin", OneHotEncoder(drop="if_binary"), categorical_binary)
    ]
)


In [13]:
# Séparation X / y
X_train = stratified_train.drop(columns=["y"])
y_train = LabelEncoder().fit_transform(stratified_train["y"])

# Appliquer uniquement le préprocessing
X_train_transformed = preprocessor.fit_transform(X_train)

# Probabilités pour la classe positive (y = 1)
X_test = stratified_test.drop(columns=["y"])
# Appliquer uniquement le préprocessing
X_test = preprocessor.fit_transform(X_test)
y_test = LabelEncoder().fit_transform(stratified_test["y"])
